# Stage 1 — Dataset Construction (IndicCorpV2 → Language ID dataset)

Goal: build a balanced, sentence-level language identification dataset from IndicCorpV2:
- 23 languages, exact split names taken from the repo's own YAML config (so labels match the repository, as required).
- 1000 sentences per language, sampled uniformly from a streamed prefix of each language's corpus via **reservoir sampling** (so we don't have to download full files — disk only has ~14GB free).
- Paragraphs are split into sentences using a **hand-written, rule-based sentence splitter** — no external NLP library (no `indic-nlp-library`, no `nltk`, nothing) is used anywhere in this step, per the constraint that tokenization must be fully manual.
- Stratified 80/10/10 train/val/test split, done independently per language so every split has exactly the same per-language counts.


## Design choice: a manual, script-agnostic sentence splitter

No sentence-tokenization library is used. Instead we split on a fixed set of sentence-ending
punctuation characters, chosen to cover every script actually present across the 23 languages:

| Character | Used by |
|---|---|
| `।` (danda, U+0964) and `॥` (double danda, U+0965) | Devanagari-family scripts (Hindi, Marathi, Nepali, Sanskrit, Bodo, Dogri, Konkani, Maithili, …) and other Brahmi-derived scripts that borrow the danda convention |
| `.`, `!`, `?` | Latin-style punctuation — used natively by Khasi (Latin script) and, in practice, mixed into modern digitized text across nearly every language here |
| `۔` (Arabic full stop, U+06D4) and `؟` (Arabic question mark, U+061F) | Perso-Arabic scripts (Urdu, Kashmiri, and — as the Stage 1 script check below confirms — the `snd_Deva` split, which is actually Perso-Arabic despite its label) |
| `…` (ellipsis) | treated as a sentence boundary wherever it appears |

a `.` is *not* treated as a sentence boundary
when it sits between two digits (e.g. `3.14`), so we don't shred decimal numbers. Beyond that we
do **not** attempt abbreviation detection (e.g. "Dr.", "इत्यादि.") — a real NLP sentence splitter
would have logic for this, but building that by hand reliably across 23 languages' worth of
abbreviation conventions is its own research project. The honest trade-off: a small fraction of
sentences may be split slightly early after an abbreviation. This is documented here rather than
hidden, and the length filter in the collection step (20–300 characters) discards the tiniest
fragments that this would otherwise produce.

Because this splitter is purely punctuation-driven and not script-aware, it needs **no per-language
routing table** — the same function runs on all 23 languages.


In [1]:
import time
import random
import hashlib
import json
import os
import re

import pandas as pd
from datasets import load_dataset

RANDOM_SEED = 42
DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)


In [2]:
# 23 official split names, exactly as given in the ai4bharat/IndicCorpV2 YAML config
LANGUAGES = [
    "asm_Beng", "ben_Beng", "brx_Deva", "doi_Deva", "gom_Deva", "guj_Gujr",
    "hin_Deva", "kan_Knda", "kas_Arab", "mai_Deva", "mal_Mlym", "mar_Deva",
    "mni_Mtei", "npi_Deva", "ory_Orya", "pan_Guru", "san_Deva", "snd_Deva",
    "tam_Taml", "tel_Telu", "urd_Arab", "khasi", "santhali",
]
assert len(LANGUAGES) == 23


In [ ]:
SENTENCE_END_CHARS = "।॥.!?۔؟…"
_SENT_END_SET = set(SENTENCE_END_CHARS)


def sentence_split(text):
    text = text.strip()
    if not text:
        return []

    sentences = []
    start = 0
    n = len(text)
    for i, ch in enumerate(text):
        if ch not in _SENT_END_SET:
            continue
        # do not split a decimal number like "3.14" on the period
        if ch == "." and 0 < i < n - 1 and text[i - 1].isdigit() and text[i + 1].isdigit():
            continue
        piece = text[start:i + 1].strip()
        if piece:
            sentences.append(piece)
        start = i + 1

    tail = text[start:].strip()
    if tail:
        sentences.append(tail)

    return sentences


# quick demo
demo = "यह पहला वाक्य है। यह दूसरा वाक्य है, जिसमें 3.14 जैसी संख्या भी है. Is this a third sentence? Yes!"
print(sentence_split(demo))


['यह पहला वाक्य है।', 'यह दूसरा वाक्य है, जिसमें 3.14 जैसी संख्या भी है.', 'Is this a third sentence?', 'Yes!']


In [4]:
def collect_sentences(split_name, target=1000, min_len=20, max_len=300,
                       max_paragraphs=15000, seed=RANDOM_SEED):
    """Reservoir-sample `target` sentences from a streamed prefix of a language split."""
    rng = random.Random(seed)
    ds = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2", split=split_name, streaming=True)

    reservoir = []
    seen_hashes = set()
    n_paragraphs = 0
    n_valid_seen = 0  # count of valid, non-duplicate sentences seen (drives reservoir replacement)

    for row in ds:
        n_paragraphs += 1
        text = row["text"].strip()
        if text:
            for sent in sentence_split(text):
                sent = sent.strip()
                if not (min_len <= len(sent) <= max_len):
                    continue
                h = hashlib.md5(sent.encode("utf-8")).hexdigest()
                if h in seen_hashes:
                    continue
                seen_hashes.add(h)

                n_valid_seen += 1
                if len(reservoir) < target:
                    reservoir.append(sent)
                else:
                    j = rng.randint(0, n_valid_seen - 1)
                    if j < target:
                        reservoir[j] = sent
        if n_paragraphs >= max_paragraphs:
            break

    return reservoir, {"n_paragraphs_scanned": n_paragraphs, "n_valid_unique_sentences": n_valid_seen}


In [5]:
results = {}
stats = {}
t_start = time.time()

for lang in LANGUAGES:
    t0 = time.time()
    sents, s = collect_sentences(lang)
    dt = time.time() - t0
    results[lang] = sents
    stats[lang] = {**s, "n_collected": len(sents), "seconds": round(dt, 1)}
    print(f"{lang:12s} collected={len(sents):4d}/1000  "
          f"paragraphs_scanned={s['n_paragraphs_scanned']:6d}  "
          f"valid_unique_seen={s['n_valid_unique_sentences']:6d}  ({dt:.1f}s)")

print(f"\nTotal time: {time.time() - t_start:.1f}s")


asm_Beng     collected=1000/1000  paragraphs_scanned= 15000  valid_unique_seen= 15932  (19.3s)


ben_Beng     collected=1000/1000  paragraphs_scanned= 15000  valid_unique_seen= 27420  (13.6s)


brx_Deva     collected=1000/1000  paragraphs_scanned= 15000  valid_unique_seen= 12157  (11.3s)


doi_Deva     collected=1000/1000  paragraphs_scanned= 12342  valid_unique_seen=  7391  (4.6s)


gom_Deva     collected=1000/1000  paragraphs_scanned= 15000  valid_unique_seen= 16096  (8.5s)


guj_Gujr     collected=1000/1000  paragraphs_scanned= 15000  valid_unique_seen= 21102  (9.1s)


... (13 outputs left out)

tel_Telu     collected=1000/1000  paragraphs_scanned= 15000  valid_unique_seen= 27360  (10.8s)


urd_Arab     collected=1000/1000  paragraphs_scanned= 15000  valid_unique_seen= 20681  (8.3s)


khasi        collected=1000/1000  paragraphs_scanned= 15000  valid_unique_seen=  9968  (6.2s)


santhali     collected=1000/1000  paragraphs_scanned= 15000  valid_unique_seen=  6522  (7.5s)

Total time: 230.3s


## Checkpoint: verify every language reached the 1000-sentence target

If any language fell short (small corpus, or too strict a length filter), it'll show `n_collected < 1000` below.


In [ ]:
short = {lang: s["n_collected"] for lang, s in stats.items() if s["n_collected"] < 1000}
if short:
    print("Languages below target of 1000 sentences:")
    for lang, n in short.items():
        print(f"  {lang}: {n}")
else:
    print("All 23 languages reached the full 1000-sentence target.")

with open(os.path.join(DATA_DIR, "collection_stats.json"), "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)


All 23 languages reached the full 1000-sentence target.


## Script sanity check

Before finalizing, verify each label's sampled text is actually written in the script its name
implies — this is how we caught (in the prior version of this pipeline) that `snd_Deva` is
mislabeled: the repo calls it "Deva" but the sampled content is 100% Perso-Arabic script.


In [7]:
SCRIPT_RANGES = {
    "Deva": ("\u0900", "\u097f"), "Beng": ("\u0980", "\u09ff"), "Gujr": ("\u0a80", "\u0aff"),
    "Guru": ("\u0a00", "\u0a7f"), "Orya": ("\u0b00", "\u0b7f"), "Taml": ("\u0b80", "\u0bff"),
    "Telu": ("\u0c00", "\u0c7f"), "Knda": ("\u0c80", "\u0cff"), "Mlym": ("\u0d00", "\u0d7f"),
    "Arab": ("\u0600", "\u06ff"), "Mtei": ("\uabc0", "\uabff"), "Olck": ("\u1c50", "\u1c7f"),
}

def dominant_script(sample_texts):
    counts = {name: 0 for name in SCRIPT_RANGES}
    counts["Latn/other"] = 0
    for text in sample_texts:
        for ch in text:
            matched = False
            for name, (lo, hi) in SCRIPT_RANGES.items():
                if lo <= ch <= hi:
                    counts[name] += 1
                    matched = True
                    break
            if not matched and ch.isalpha():
                counts["Latn/other"] += 1
    total = sum(counts.values()) or 1
    return {k: round(v / total, 3) for k, v in counts.items() if v > 0}

for lang in LANGUAGES:
    print(f"{lang:12s} {dominant_script(results[lang][:30])}")


asm_Beng     {'Deva': 0.009, 'Beng': 0.988, 'Latn/other': 0.002}
ben_Beng     {'Deva': 0.015, 'Beng': 0.983, 'Latn/other': 0.002}
brx_Deva     {'Deva': 0.999, 'Latn/other': 0.001}
doi_Deva     {'Deva': 1.0}
gom_Deva     {'Deva': 0.97, 'Latn/other': 0.03}
guj_Gujr     {'Gujr': 1.0}
hin_Deva     {'Deva': 0.996, 'Latn/other': 0.004}
kan_Knda     {'Knda': 0.996, 'Latn/other': 0.004}
kas_Arab     {'Arab': 0.999, 'Latn/other': 0.001}
mai_Deva     {'Deva': 1.0}
mal_Mlym     {'Mlym': 0.998, 'Latn/other': 0.002}
mar_Deva     {'Deva': 0.994, 'Latn/other': 0.006}
mni_Mtei     {'Mtei': 0.939, 'Latn/other': 0.061}
npi_Deva     {'Deva': 1.0}
ory_Orya     {'Deva': 0.012, 'Orya': 0.988}
pan_Guru     {'Deva': 0.009, 'Guru': 0.991}
san_Deva     {'Deva': 1.0}
snd_Deva     {'Arab': 1.0}
tam_Taml     {'Taml': 0.987, 'Latn/other': 0.013}
tel_Telu     {'Telu': 1.0}
urd_Arab     {'Arab': 0.99, 'Latn/other': 0.01}
khasi        {'Latn/other': 1.0}
santhali     {'Deva': 0.001, 'Olck': 0.957, 'Latn/other': 0.043}

## Stratified 80/10/10 split

Done independently per language: shuffle that language's 1000 sentences (fixed seed), then slice
800/100/100. Concatenating across languages guarantees every split has exactly the same per-language
sentence count.


In [8]:
train_rows, val_rows, test_rows = [], [], []
rng = random.Random(RANDOM_SEED)

for lang in LANGUAGES:
    sents = results[lang][:]
    rng.shuffle(sents)
    n = len(sents)
    n_train = int(round(n * 0.8))
    n_val = int(round(n * 0.1))
    # remainder goes to test, so counts always sum to n even if n != 1000
    train_part = sents[:n_train]
    val_part = sents[n_train:n_train + n_val]
    test_part = sents[n_train + n_val:]

    train_rows += [{"text": s, "label": lang} for s in train_part]
    val_rows += [{"text": s, "label": lang} for s in val_part]
    test_rows += [{"text": s, "label": lang} for s in test_part]

train_df = pd.DataFrame(train_rows)
val_df = pd.DataFrame(val_rows)
test_df = pd.DataFrame(test_rows)

print("train:", train_df.shape, " val:", val_df.shape, " test:", test_df.shape)


train: (18400, 2)  val: (2300, 2)  test: (2300, 2)


In [9]:
# Sanity check: per-language counts in each split
counts = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "val": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).fillna(0).astype(int).loc[LANGUAGES]
counts


,train,val,test
label,,,
asm_Beng,800,100,100
ben_Beng,800,100,100
brx_Deva,800,100,100
doi_Deva,800,100,100
gom_Deva,800,100,100
guj_Gujr,800,100,100
hin_Deva,800,100,100
kan_Knda,800,100,100
kas_Arab,800,100,100


In [10]:
train_df.to_csv(os.path.join(DATA_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(DATA_DIR, "val.csv"), index=False)
test_df.to_csv(os.path.join(DATA_DIR, "test.csv"), index=False)
print("Saved train.csv, val.csv, test.csv to", os.path.abspath(DATA_DIR))


Saved train.csv, val.csv, test.csv to /home/deeraj/LLM/Lab1/data


## Spot-check: a few sampled sentences per language

In [11]:
for lang in LANGUAGES:
    example = results[lang][0] if results[lang] else "<none collected>"
    print(f"{lang:12s}: {example}")


asm_Beng    : মাণিকপুৰ: মাণিকপুৰত অবৈধ অৰুণাচলী সুৰা ভৰ্ত্তি ট্ৰাক জব্দ।
ben_Beng    : তবে দর্শকদের কাছ থেকে যে ভালোবাসা পেয়েছেন, তা ২১ গ্র্যান্ড স্লাম জয়ের সমান বললেন জোকোভিচ।
brx_Deva    : बे दायजों लोबबा थानायनि सन्देहआव गासै 32 जोन सुबुंफोरखौ पुलिसआ हमदोंमोन।
doi_Deva    : जान देने पर तुले दा की ऐ कोई
gom_Deva    : दुसऱ्या दिवशी पहाटे पहाटे च जाग आली आणि सबंध गुहागर चा फेर फटका उरकुन आलो.
guj_Gujr    : આ બ્રાઝીલ નટના સૌથી મોટા એક્સપોર્ટર તરીકે બોલિવિયાનું નામ ટોચ પર આવે છે.
hin_Deva    : रात के खाने में भूसे से निकाला गया पका हुआ आम और रोटी .
kan_Knda    : ನೀವು ಸ್ವಯಂ ತೊಡಗಿಸಿಕೊಳ್ಳಲು ಮೊದಲು, ನೀವು ಕಂಡುಹಿಡಿಯಬೇಕು ಗುಳ್ಳೆಗಳನ್ನು ಇದ್ದವು ಏಕೆ ಮುಖದ ಮೇಲೆ.
kas_Arab    : ویلیٹ چھِ تِتھَے کٔن اِستِمال کٔرِتھ ہیکان یِتھٕ کٔنی اَکھ کریڈِٹ کارڈ یا ڈیبِٹ کارڈ کَران چھِ۔
mai_Deva    : भीतरसँ विदाह भ’ गेल अछि
mal_Mlym    : ഒരു ദശാബ്ദത്തോളം യേശുദാസിന്റെ കച്ചേരികളിലും മറ്റും വയലിനിലെ സ്ഥിരം സാന്നിധ്യം.
mar_Deva    : त्यामुळे विजयाची हीच लय कायम राखत भारतीय संघ दुसऱ्या मालिकेत विजय मिळवण्यासाठी प्रयत